In [ ]:
# RONALD WILLIAM D. GERON
# CPE3B

In [1]:
!pip install openai

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 17.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 28.7 MB/s  0:00:00

   ---- ----------------------------------- 1/9 [tqdm]
   ---- ----------------------------------- 1/9 [tqdm]
   ---- ----------------------------------- 1/9 [tqdm]
   ---- ----------------------------------- 1/9 [tqdm]
   ---- ----------------------------------- 1/9 [tqdm]
   ---- ----------------------------------- 1/9 [tqdm]
   ---- ----------------------------------- 1/9 [tqdm]
   ---- ----------------------------------- 1/9 [tqdm]
   -------- ------------------------------- 2/9 [sniffio]
   ------------- -------------------------- 3/9 [pydantic-core]
   ---------------------- ----------------- 5/9 [distro]
   ---------------------- ----------------- 5/9 [distro]
   ---------------------- -----------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from openai import OpenAI

# Connect to LM Studio local server
client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio"  # any string works
)

In [ ]:
# Send a simple prompt
response = client.chat.completions.create(
    model="tinyllama-1.1b-chat-v1.0",  # name can be anything in LM Studio
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain what a microcontroller is."}
    ],
    temperature=0.7
)

# Print response
print(response.choices[0].message.content)

A microcontroller (MCU) is a small, compact integrated circuit that is designed to perform specific tasks such as controlling and processing data. Unlike traditional digital computers or electronic circuits, MCUs are designed to be highly programmable and can run specialized software code that performs various functions.

A typical MCU has a single chip that is typically made of silicon and contains many microprocessors (usually called cores), memory, and other components required for its intended purpose. These microprocessors can execute various instructions at a high speed, allowing the MCU to perform complex tasks quickly and efficiently.

A typical MCU may include features such as:

1. Programmable logic: The MCU can be programmed by writing code that controls its behavior. This allows the device to perform specific tasks or set up routines for specific applications.

2. Memory: The MCU has a large amount of non-volatile memory, which means it is not affected by power outages or o

In [29]:
import keyboard
from langchain_community.chat_message_histories import ChatMessageHistory

# Initialize the history object to keep track of the discussion
history = ChatMessageHistory()

In [5]:
import keyboard
from langchain_community.chat_message_histories import ChatMessageHistory
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio"
)

history = ChatMessageHistory()
user_name = None  # store name if introduced

system_anchor = {
    "role": "system",
    "content": (
        "You are a helpful assistant in a continuous conversation. "
        "Always answer based only on the conversation history. "
        "If the user says 'My name is X', remember it and when asked "
        "'What is my name?' reply directly with 'Your name is X'. "
        "Do not invent a persona or add disclaimers."
    )
}

while True:
    query = input("\nUser (Enter to Continue || Type 'exit' or press ESC to exit): ")

    if keyboard.is_pressed('Esc') or query.lower() == 'exit':
        print("\nExiting...")
        print("Program Finished.")
        break

    print("User Input:", query)
    print('- ' * 50)

    if not query.strip():
        continue

    # Detect if user introduces their name
    if "my name is" in query.lower():
        user_name = query.split("is", 1)[1].strip()

    # Fallback: if user asks directly
    if "what is my name" in query.lower() and user_name:
        print("\nAssistant: Your name is", user_name)
        history.add_ai_message(f"Your name is {user_name}")
        print("\n" + "=" * 100 + "\n")
        continue

    # Add user message to LangChain history
    history.add_user_message(query)

    # Convert LangChain history to OpenAI format
    messages_with_history = [system_anchor] + [
        {"role": "user" if m.type == "human" else "assistant", "content": m.content}
        for m in history.messages
    ]

    # Send the full history to LM Studio
    stream = client.chat.completions.create(
        model="tinyllama-1.1b-chat-v1.0",
        messages=messages_with_history,
        stream=True
    )

    # Collect the streamed response
    full_ai_response = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            content = chunk.choices[0].delta.content
            print(content, end="", flush=True)
            full_ai_response += content

    # Save the AI response back to LangChain history
    history.add_ai_message(full_ai_response)

    print("\n")
    print("=" * 100)
    print(" ")


User Input: hello, my name is william
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
My pleasure! Your name is William. That's great to hear! Please feel free to ask me any questions during our conversation. I can always provide accurate responses based on the historical context in which you may have encountered this particular name. Let me know if there's anything else I can assist you with. Best regards,
[Your Name]

 
User Input: what is the capital of the Philippines?
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
The capital city of the Philippines is Manila. The official name of the country is "Republic of the Philippines." Please let me know if there's anything else I can assist you with. Best regards,
[Your Name]

 
User Input: what is my name again?
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

Assistant: Your name 